In [90]:
import torch
import numpy as np

from utils import *

In [91]:
if torch.cuda.is_available():
    device = 'cuda:0'
else:
    device = 'cpu'

# dataset
dataset = "syn"
print(dataset)

d1 = 10000
d2 = 1000
r = 10
M = load_data_syn(r, d1, d2, device)

# privacy
delta = 1/d1
epsilon = 10
sigma = torch.sqrt(2*torch.log(torch.tensor(1.25/delta)))/epsilon

# main part
err_list = []
rmse_list = []    
p = 0.5

# sample 2 entries per row
#observed_M, masks = get_random_samples_per_row(M.cpu().numpy(), 2)
#observed_M = torch.from_numpy(observed_M).float().to(device)
#masks = torch.from_numpy(masks).to(device)

# IID sample
observed_M, masks = get_uniform_masks(M, p)

# observed second-moment matrix
MTM = M.T @ M
cov_observe_M =  observed_M.T @ observed_M
T_masks = 1 * (cov_observe_M!=0)

syn


In [101]:
# True probability weighting
diag_cov = torch.diag( torch.diag(cov_observe_M) )
T_p = ((1.0 / p) * diag_cov + (1.0 / (p**2)) * (cov_observe_M - diag_cov))
print("relative error of T_p: ", torch.norm(T_p*T_masks - MTM*T_masks)/torch.norm(MTM*T_masks))

relative error of T_p:  tensor(0.0171, device='cuda:0')


In [100]:
# single value estimated probability weighting
p_hat = (observed_M!=0).sum()/(d1*d2)
T_single_p_hat = ((1.0 / p_hat) * diag_cov + (1.0 / (p_hat**2)) * (cov_observe_M - diag_cov))
print("relative error of T_single_p_hat: ", torch.norm(T_single_p_hat*T_masks - MTM*T_masks)/torch.norm(MTM*T_masks))

relative error of T_single_p_hat:  tensor(0.0171, device='cuda:0')


In [102]:
print(p)
print(p_hat)

0.5
tensor(0.4999, device='cuda:0')


In [95]:
# Inverse estimated probability weighting in matrix
cov_observe_count = (1 * (observed_M != 0)).float().T @ (1 * (observed_M != 0).float())
cov_observe_count = cov_observe_count + (cov_observe_count == 0) * 1
p_hat_matrix = cov_observe_count/d1

T = cov_observe_M / p_hat_matrix

# calculate matrix of p_hat and p
p_hat_matrix = (cov_observe_count/d1)
p_matrix = (torch.diag(torch.diag(torch.ones(d2, d2)*(p-p**2))) + torch.ones(d2, d2)*p**2).to(device)

# calculate (p_hat - p)
print("mean value of (p_hat - p) in matrix: ", (p_matrix*T_masks-p_hat_matrix*T_masks).mean()/(T_masks.sum() / d2**2))
# calculate the relative err of T_p_hat
print("relative error of T_p_hat: ", torch.norm(T*T_masks-MTM*T_masks)/torch.norm(MTM*T_masks))

mean value of (p_hat - p) in matrix:  tensor(0.0001, device='cuda:0')
relative error of T_p_hat:  tensor(0.0016, device='cuda:0')


In [96]:
print(p_hat_matrix)

tensor([[0.5024, 0.2515, 0.2578,  ..., 0.2466, 0.2506, 0.2511],
        [0.2515, 0.4988, 0.2501,  ..., 0.2535, 0.2514, 0.2500],
        [0.2578, 0.2501, 0.5089,  ..., 0.2514, 0.2552, 0.2584],
        ...,
        [0.2466, 0.2535, 0.2514,  ..., 0.4993, 0.2484, 0.2510],
        [0.2506, 0.2514, 0.2552,  ..., 0.2484, 0.4978, 0.2507],
        [0.2511, 0.2500, 0.2584,  ..., 0.2510, 0.2507, 0.5007]],
       device='cuda:0')


In [97]:
print(p_matrix)

tensor([[0.5000, 0.2500, 0.2500,  ..., 0.2500, 0.2500, 0.2500],
        [0.2500, 0.5000, 0.2500,  ..., 0.2500, 0.2500, 0.2500],
        [0.2500, 0.2500, 0.5000,  ..., 0.2500, 0.2500, 0.2500],
        ...,
        [0.2500, 0.2500, 0.2500,  ..., 0.5000, 0.2500, 0.2500],
        [0.2500, 0.2500, 0.2500,  ..., 0.2500, 0.5000, 0.2500],
        [0.2500, 0.2500, 0.2500,  ..., 0.2500, 0.2500, 0.5000]],
       device='cuda:0')


In [98]:
# calculate Taylor expansion in diagonal elements
ratio = T_masks.sum() / d2**2
diag_cov = torch.diag( torch.diag(cov_observe_M) )

T_p_1_diag = (-1.0 / p**2) * diag_cov
T_p_2_diag = (1.0 / p**3) * diag_cov 
T_p_3_diag = (-1.0 / p**4) * diag_cov 

item_1 = torch.mul(T_p_1_diag,(p_hat_matrix-p_matrix).diag())
item_2 = torch.mul(T_p_2_diag,(p_hat_matrix-p_matrix).diag()**2)
item_3 = torch.mul(T_p_3_diag,(p_hat_matrix-p_matrix).diag()**3)

#mask_err_Tp = T_p[T_masks] - MTM[T_masks]
#print(MTM)
#print(T_p)
#print(T_p_1)
#print(T_p_2)

# the result of each item in the Taylor expansion
#num = (diag_cov !=0).sum()
num = (T_masks!=0).sum()
print("value of 1st-order item in diag: ", (item_1*T_masks).sum()/num)
print("value of 2nd-order item in diag: ", (item_2*T_masks).sum()/num)
print("value of 2th-order item in diag: ", (item_3*T_masks).sum()/num)

value of 1st-order item in diag:  tensor(0.0046, device='cuda:0')
value of 2nd-order item in diag:  tensor(0.0039, device='cuda:0')
value of 2th-order item in diag:  tensor(3.2439e-06, device='cuda:0')


In [99]:
# calculate Taylor expansion in off-diag elements
ratio = T_masks.sum() / d2**2
diag_cov = torch.diag( torch.diag(cov_observe_M) )
T_p_1_offdiag = - (2.0 / (p**3)) * (cov_observe_M - diag_cov)
T_p_2_offdiag = (3.0 / (p**4)) * (cov_observe_M - diag_cov)
T_p_3_offdiag =  - (4.0 / (p**5)) * (cov_observe_M - diag_cov)

delta_p_offdiag = (p_hat_matrix-p_matrix).fill_diagonal_(0)
item_1 = torch.mul(T_p_1_offdiag,delta_p_offdiag)
item_2 = torch.mul(T_p_2_offdiag,delta_p_offdiag**2)
item_3 = torch.mul(T_p_3_offdiag,delta_p_offdiag**3)

#mask_err_Tp = T_p[T_masks] - MTM[T_masks]
#print(MTM)
#print(T_p)
#print(T_p_1)
#print(T_p_2)

# the result of each item in the Taylor expansion
#num = T_masks.sum() - (diag_cov !=0).sum()
num = (T_masks!=0).sum()
print("value of 1st-order item: ", (item_1*T_masks).sum()/num)
print("value of 2nd-order item: ", (item_2*T_masks).sum()/num)
print("value of 2th-order item: ", (item_3*T_masks).sum()/num)

value of 1st-order item:  tensor(4.9982, device='cuda:0')
value of 2nd-order item:  tensor(8.7632, device='cuda:0')
value of 2th-order item:  tensor(0.0026, device='cuda:0')
